In [1]:
import pandas as pd

TASK3_40K_PATH = "/content/drive/MyDrive/CORTEX/T3/task3_strong_modality_40k.csv"
df = pd.read_csv(TASK3_40K_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

Shape: (40012, 25)
Columns: ['Unique_ID', 'age_group', 'tier_sim', 'I_present', 'I_severity_level', 'I_freq', 'I_duration', 'I_controllability', 'I_deterrents', 'I_reasons', 'B_preparatory', 'B_aborted', 'B_interrupted', 'B_actual_attempt', 'B_any', 'days_since_last_event', 'duration_since_onset', 'points_behavior', 'points_ideation', 'points_intensity', 'points_acute', 'points_persistence', 'points_temporal', 'severity_score_rule', 'risk_band']


,Unique_ID,age_group,tier_sim,I_present,I_severity_level,I_freq,I_duration,I_controllability,I_deterrents,I_reasons,...,days_since_last_event,duration_since_onset,points_behavior,points_ideation,points_intensity,points_acute,points_persistence,points_temporal,severity_score_rule,risk_band
0,0,Adult,0,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Low
1,1,Teen,1,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Low
2,2,Adult,2,1,4,3,2,4,3,4,...,28,48,0.0,28.0,11.0,4.0,4.0,8.0,47.0,Medium


In [2]:
import numpy as np

# ---- Targets ----
y_score = df["severity_score_rule"].astype(float)     # regression target (0-100)
y_band  = df["risk_band"].astype(str)                # optional classification target

# ---- Non-leaky feature columns (raw intake only) ----
feature_cols = [
    "age_group",
    "I_present", "I_severity_level", "I_freq", "I_duration", "I_controllability",
    "I_deterrents", "I_reasons",
    "B_preparatory", "B_aborted", "B_interrupted", "B_actual_attempt", "B_any",
    "days_since_last_event", "duration_since_onset",
]

missing = [c for c in feature_cols if c not in df.columns]
print("Missing:", missing)

X = df[feature_cols].copy()

# Encode age_group
X["age_group"] = X["age_group"].map({"Teen": 1, "Adult": 0}).fillna(0).astype(int)

# Clean numeric
for c in X.columns:
    X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)

print("X shape:", X.shape)
X.head(3)

Missing: []
X shape: (40012, 15)


,age_group,I_present,I_severity_level,I_freq,I_duration,I_controllability,I_deterrents,I_reasons,B_preparatory,B_aborted,B_interrupted,B_actual_attempt,B_any,days_since_last_event,duration_since_onset
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,1,4,3,2,4,3,4,0,0,0,0,0,28,48


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, band_train, band_test = train_test_split(
    X, y_score, y_band,
    test_size=0.2,
    random_state=42,
    stratify=y_band
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train band dist:\n", band_train.value_counts())

Train: (32009, 15) Test: (8003, 15)
Train band dist:
 risk_band
Low         19313
Critical     5769
Medium       5738
High         1189
Name: count, dtype: int64


In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

reg = RandomForestRegressor(
    n_estimators=400,
    random_state=42,
    min_samples_leaf=2,
    n_jobs=-1
)

reg.fit(X_train, y_train)

pred = reg.predict(X_test)
pred = np.clip(pred, 0, 100)

print("MAE:", mean_absolute_error(y_test, pred))
print("R^2:", r2_score(y_test, pred))

MAE: 0.3194100581476406
R^2: 0.9995825111119706


In [5]:
def score_to_band(score: float) -> str:
    # Example thresholds (adjust to your final)
    if score < 25: return "Low"
    if score < 50: return "Medium"
    if score < 75: return "High"
    return "Critical"

pred_band = pd.Series(pred).apply(score_to_band)

from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(band_test, pred_band, labels=["Low","Medium","High","Critical"]))
print(classification_report(band_test, pred_band, digits=4))

[[4407  422    0    0]
 [   0 1068  367    0]
 [   0    0  264   33]
 [   0    0   13 1429]]
              precision    recall  f1-score   support

    Critical     0.9774    0.9910    0.9842      1442
        High     0.4099    0.8889    0.5611       297
         Low     1.0000    0.9126    0.9543      4829
      Medium     0.7168    0.7443    0.7303      1435

    accuracy                         0.8957      8003
   macro avg     0.7760    0.8842    0.8075      8003
weighted avg     0.9233    0.8957    0.9049      8003



In [6]:
import joblib, os

OUT_DIR = "/content/drive/MyDrive/CORTEX/T3/models"
os.makedirs(OUT_DIR, exist_ok=True)

joblib.dump(reg, f"{OUT_DIR}/task3_score_regressor.pkl")
joblib.dump(feature_cols, f"{OUT_DIR}/task3_feature_cols.pkl")

print("Saved to:", OUT_DIR)

Saved to: /content/drive/MyDrive/CORTEX/T3/models
